# Notebook for creating plots of cluster summaries
Needs to be run after `..._clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path(os.getcwd()), *Path(os.getcwd()).parents] if path.stem == "esa_consulting")
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
import ast
from pathlib import Path
from typing import cast

import pandas as pd
import plotnine as p9
from IPython.display import display

from utils.clustering_utils import cluster_summaries
from utils.misc_utils import display_side_by_side, value_counts
from utils.plotting_utils import make_plot_categorical_var_dist_by_cluster, make_plot_continuous_var_dist_by_cluster
from utils.saving_utils import DatasetBundle, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
data_discov_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
data_discov_full.meta = cast(pd.DataFrame, data_discov_full.meta)
data_discov_full.na = cast(pd.DataFrame, data_discov_full.na)
data_valid_full = DatasetBundle.load(Path("data/preprocessed/data_valid_full"))
data_valid_full.meta = cast(pd.DataFrame, data_valid_full.meta)
data_valid_full.na = cast(pd.DataFrame, data_valid_full.na)

metrics_grid_search_discov = pd.read_parquet(Path("outputs/gmm_clustering/metrics_grid_search_discov.parquet"))

proba_and_labels_discov = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_discov.parquet"))
proba_and_labels_valid = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_valid.parquet"))

# Set outputs folder
outputs_folder = "outputs/gmm_clustering"

# Set predicted probability and labels
proba_discov = proba_and_labels_discov.drop("label", axis="columns")
proba_valid = proba_and_labels_valid.drop("label", axis="columns")
labels_discov = proba_and_labels_discov["label"]
labels_valid = proba_and_labels_valid["label"]

# Set sample weights
sample_weight_discov = data_discov_full.data["wichtn"].to_numpy()
sample_weight_valid = data_valid_full.data["wichtn"].to_numpy()

In [ ]:
# Print cluster proportions
proportions_discov = (
    value_counts(labels_discov, sort=True, normalize=True, dropna=False, sample_weight=sample_weight_discov)
    .apply(lambda x: f"{x:.2%}")
    .to_frame()
    .reset_index(names="cluster")
)
proportions_valid = (
    value_counts(labels_valid, sort=True, normalize=True, dropna=False, sample_weight=sample_weight_valid)
    .apply(lambda x: f"{x:.2%}")
    .to_frame()
    .reset_index(names="cluster")
)
display_side_by_side(proportions_discov, proportions_valid, captions=["discov", "valid"])

In [ ]:
# Interpretation of variables in the discovery dataset
cluster_summaries_not_nominal_discov, cluster_summaries_nominal_discov = cluster_summaries(
    data_discov_full.drop(["lfd", "fall"]), labels_discov, sample_weight=sample_weight_discov
)

print("Summaries of continuous, binary, and ordinal variables in the discovery dataset:")
display(cluster_summaries_not_nominal_discov)

print("Summaries of nominal variables in the discovery dataset:")
display(cluster_summaries_nominal_discov)

In [ ]:
# Interpretation of variables in the validation dataset
cluster_summaries_not_nominal_valid, cluster_summaries_nominal_valid = cluster_summaries(
    data_valid_full.drop(["lfd", "fall"]), labels_valid, sample_weight=sample_weight_valid
)

print("Summaries of continuous, binary, and ordinal variables in the validation dataset:")
display(cluster_summaries_not_nominal_valid)

print("Summaries of nominal variables in the validation dataset:")
display(cluster_summaries_nominal_valid)

In [ ]:
# Plot distribution of continuous variable in different clusters
var = "Anzahl"
quantiles = [0.02, 0.1, 0.33, 0.5, 0.67, 0.9, 0.98]

# Make plots
continuous_var_plot_discov = make_plot_continuous_var_dist_by_cluster(
    data_discov_full.data,
    labels_discov,
    var=var,
    quantiles=quantiles,
    sample_weight=sample_weight_discov,
    sort=True,
    ascending=False,
    cluster_size_decimals=1,
    column_size=7,
)
continuous_var_plot_valid = make_plot_continuous_var_dist_by_cluster(
    data_valid_full.data,
    labels_valid,
    var=var,
    quantiles=quantiles,
    sample_weight=sample_weight_valid,
    sort=True,
    ascending=False,
    cluster_size_decimals=1,
    column_size=7,
)

# Combine plots
continuous_var_plot_all = (
    continuous_var_plot_discov + p9.labs(title="Discovery") + continuous_var_plot_valid + p9.labs(title="Validation")
) & p9.theme(figure_size=(16, 6))
continuous_var_plot_all.show()

# Save plots
save_plot(continuous_var_plot_discov, Path(outputs_folder, "continuous_var_plot_discov.png"), dpi=150)
save_plot(continuous_var_plot_valid, Path(outputs_folder, "continuous_var_plot_valid.png"), dpi=150)

# save_plot(continuous_var_plot_all, Path(outputs_folder, "continuous_var_plot_all.png"), dpi=150)

In [ ]:
# Plot distribution of categorical variable in different clusters
var = "erwerb"
var_value_labels = ast.literal_eval(data_discov_full.meta.loc[lambda x: x["name"].eq(var), "labels"].item())

# Make plots
categorical_var_plot_discov = make_plot_categorical_var_dist_by_cluster(
    data_discov_full.data,
    labels_discov,
    var=var,
    sample_weight=sample_weight_discov,
    cluster_size_decimals=1,
    value_labels=var_value_labels,
)
categorical_var_plot_valid = make_plot_categorical_var_dist_by_cluster(
    data_valid_full.data,
    labels_valid,
    var=var,
    sample_weight=sample_weight_valid,
    cluster_size_decimals=1,
    value_labels=var_value_labels,
)

# Combine plots
categorical_var_plot_all = (
    categorical_var_plot_discov + p9.labs(title="Discovery") + categorical_var_plot_valid + p9.labs(title="Validation")
) & p9.theme(figure_size=(16, 6))
categorical_var_plot_all.show()

# Save plots
save_plot(categorical_var_plot_discov, Path(outputs_folder, "categorical_var_plot_discov.png"), dpi=150)
save_plot(categorical_var_plot_valid, Path(outputs_folder, "categorical_var_plot_valid.png"), dpi=150)

# save_plot(categorical_var_plot_all, Path(outputs_folder, "categorical_var_plot_all.png"), dpi=150)

In [ ]:
# Save summaries
proportions_discov.to_parquet(Path(outputs_folder, "proportions_discov.parquet"), index=False)
proportions_valid.to_parquet(Path(outputs_folder, "proportions_valid.parquet"), index=False)

cluster_summaries_not_nominal_discov.to_parquet(
    Path(outputs_folder, "cluster_discov_summaries_not_nominal.parquet"), index=False
)
cluster_summaries_nominal_discov.to_parquet(
    Path(outputs_folder, "cluster_discov_summaries_nominal.parquet"), index=False
)
cluster_summaries_not_nominal_valid.to_parquet(
    Path(outputs_folder, "cluster_valid_summaries_not_nominal.parquet"), index=False
)
cluster_summaries_nominal_valid.to_parquet(Path(outputs_folder, "cluster_valid_summaries_nominal.parquet"), index=False)